# 18 — Softmax, Cross-Entropy, and Numerical Stability

**Master syllabus coverage:** V2 2.9–2.10

## Why this matters

This is the core next-token objective. A correct but numerically naive formula can overflow even when the mathematically equivalent stable version is well behaved.

## Learning objectives

- Implement stable softmax, log-sum-exp, and log-softmax.
- Compute next-token cross-entropy for `[B,T,V]` logits.
- Verify the fused result against PyTorch.
- Derive and gradient-check `softmax(logits) - one_hot(target)`.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Softmax converts logits to a categorical distribution

$$\operatorname{softmax}(z)_i=\frac{e^{z_i}}{\sum_j e^{z_j}}.$$

Adding the same constant to every logit changes neither probability nor ranking.
This shift invariance permits subtracting the largest logit before exponentiation.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def stable_softmax(logits: np.ndarray, axis: int = -1) -> np.ndarray:
    shifted = logits - np.max(logits, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)

logits = np.array([[1.0, 2.0, 3.0], [1000.0, 1001.0, 1002.0]])
probabilities = stable_softmax(logits)
print(probabilities)
np.testing.assert_allclose(probabilities.sum(axis=-1), 1.0)
np.testing.assert_allclose(probabilities[0], probabilities[1])


## 2. Log-sum-exp is the stable normalizer

$$\log\sum_j e^{z_j}=m+\log\sum_j e^{z_j-m},\quad m=\max_j z_j.$$

Log-softmax is $z_i-\operatorname{LSE}(z)$. Fused cross-entropy uses log-softmax
directly instead of computing probabilities and then taking their logs.


In [ ]:
def logsumexp(x: np.ndarray, axis: int = -1, keepdims: bool = False) -> np.ndarray:
    maximum = np.max(x, axis=axis, keepdims=True)
    result = maximum + np.log(np.exp(x - maximum).sum(axis=axis, keepdims=True))
    return result if keepdims else np.squeeze(result, axis=axis)

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    return x - logsumexp(x, axis=axis, keepdims=True)

np.testing.assert_allclose(np.exp(log_softmax(logits)), probabilities)
print("stable log probabilities:\n", log_softmax(logits))


## 3. Batched next-token cross-entropy

Logits have shape `[B,T,V]`; integer targets have `[B,T]`. Flatten only after confirming
axes. For each position choose the target log-probability, negate, then average or sum.
Padding positions require an explicit mask or ignore index.


In [ ]:
rng = np.random.default_rng(42)
B, T, V = 2, 3, 5
batch_logits = rng.normal(size=(B, T, V))
targets = rng.integers(0, V, size=(B, T))
log_probs = log_softmax(batch_logits, axis=-1)
selected = np.take_along_axis(log_probs, targets[..., None], axis=-1).squeeze(-1)
manual_loss = -selected.mean()

torch_loss = F.cross_entropy(
    torch.tensor(batch_logits, dtype=torch.float64).reshape(-1, V),
    torch.tensor(targets).reshape(-1),
)
np.testing.assert_allclose(manual_loss, torch_loss.item(), rtol=1e-12)
print("cross-entropy:", manual_loss)


## 4. The elegant gradient

For one example with one-hot target $y$, softmax-cross-entropy has gradient
$\partial L/\partial z=p-y$. Increasing the target logit lowers loss; non-target
logits receive positive gradients proportional to their probability.


In [ ]:
z = torch.tensor([0.5, -1.0, 2.0], dtype=torch.float64, requires_grad=True)
target = torch.tensor([2])
loss = F.cross_entropy(z.unsqueeze(0), target)
loss.backward()
expected = torch.softmax(z.detach(), dim=-1)
expected[target] -= 1.0
torch.testing.assert_close(z.grad, expected)
print("probability minus one-hot:", z.grad)

# A finite-difference spot check.
epsilon = 1e-6
plus = z.detach().clone(); plus[0] += epsilon
minus = z.detach().clone(); minus[0] -= epsilon
numeric = (F.cross_entropy(plus[None], target) - F.cross_entropy(minus[None], target)) / (2 * epsilon)
torch.testing.assert_close(numeric, z.grad[0], atol=1e-8, rtol=1e-6)


## Failure modes to recognize

- **Exponent overflow:** raw large logits create infinity and NaN.
- **Wrong softmax axis:** probabilities normalize across time or batch rather than vocabulary.
- **Target/logit misalignment:** flattened positions no longer refer to the same tokens.
- **Padding included:** easy padding predictions distort loss and gradients.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Add a boolean padding mask and compute a mean over valid tokens only.
2. Measure probability and gradient changes at temperatures 0.5, 1, and 2.
3. Implement label smoothing manually and compare it to PyTorch cross-entropy.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can derive, implement, and gradient-check stable batched next-token cross-entropy without calling softmax first.

**Next:** 19 — Optimization.
